# Generate Chromatic Point Cloud Test Data

This notebook creates a synthetic chromatic 2D point cloud with blue points sampled near several circle boundaries and orange outlier points sampled sparsely both inside and outside the circles. The circles have different centers and radii, and they are not required to be pairwise intersecting. It saves the point cloud as a plain text file with columns:

```text
x y f
```

The scalar function value `f` follows the function-Alpha convention used here: noisy circle points have `f = 0`, and outlier/noise points have `f = 1`.

## Configuration

Adjust the parameters below to control the number of circle points, outlier points, circle centers/radii, and output path.

In [ ]:
from pathlib import Path
import os

import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / "data" / "pointcloud_examples"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MATPLOTLIB_CACHE_DIR = PROJECT_ROOT / "outputs" / "matplotlib_cache"
MATPLOTLIB_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MATPLOTLIB_CACHE_DIR))


OUTPUT_PATH = OUTPUT_DIR / "chromatic_four_circles_500pts_function.txt"

RANDOM_SEED = 20260611

TOTAL_POINTS = 500
CIRCLE_POINTS_PER_CIRCLE = 90

# Columns are center_x, center_y, radius. The circles are intentionally separated.
CIRCLE_SPECS = np.array(
    [
        [0.20, 0.20, 0.13],
        [0.82, 0.24, 0.16],
        [0.27, 0.83, 0.14],
        [0.92, 0.80, 0.18],
    ],
    dtype=float,
)
CIRCLE_CENTERS = CIRCLE_SPECS[:, :2]
CIRCLE_RADII = CIRCLE_SPECS[:, 2]
CIRCLE_POINTS = CIRCLE_POINTS_PER_CIRCLE * len(CIRCLE_SPECS)
OUTLIER_POINTS = TOTAL_POINTS - CIRCLE_POINTS

# Standard deviation for radial noise added to blue circle points.
CIRCLE_RADIAL_JITTER = 0.018

# Sparse outliers: some are inside circle disks, and the rest are outside all disks.
OUTLIER_LOCAL_FRACTION = 0.45
LOCAL_OUTLIER_POINTS = int(round(OUTLIER_POINTS * OUTLIER_LOCAL_FRACTION))
BACKGROUND_OUTLIER_POINTS = OUTLIER_POINTS - LOCAL_OUTLIER_POINTS
OUTLIER_BOUNDS = np.array([[0.02, 1.12], [0.02, 1.04]], dtype=float)

if OUTLIER_POINTS < 0:
    raise ValueError("TOTAL_POINTS must be at least CIRCLE_POINTS_PER_CIRCLE * len(CIRCLE_SPECS).")

print("Output path:", OUTPUT_PATH)

## Sample Noisy Circle Points and Sparse Outliers

Circle points are sampled by drawing angles uniformly and adding radial jitter, so they form noisy annuli instead of exact circle boundaries. Outliers are sampled sparsely from a mixture of circle disks and the background region outside all disks.

In [ ]:
def sample_circle_boundaries(
    rng: np.random.Generator,
    *,
    centers: np.ndarray,
    radii: np.ndarray,
    points_per_circle: int,
    radial_jitter: float = 0.0,
) -> np.ndarray:
    circle_points = []
    for center, radius in zip(centers, radii):
        angles = rng.uniform(0.0, 2.0 * np.pi, size=points_per_circle)
        sampled_radii = radius + rng.normal(0.0, radial_jitter, size=points_per_circle)
        sampled_radii = np.clip(sampled_radii, 0.0, None)
        offsets = np.column_stack((np.cos(angles), np.sin(angles))) * sampled_radii[:, None]
        circle_points.append(center + offsets)
    return np.vstack(circle_points)


def sample_circle_disk_outliers(
    rng: np.random.Generator,
    n_points: int,
    *,
    centers: np.ndarray,
    radii: np.ndarray,
) -> np.ndarray:
    if n_points == 0:
        return np.empty((0, 2), dtype=float)
    circle_probabilities = radii**2 / np.sum(radii**2)
    circle_indices = rng.choice(len(radii), size=n_points, p=circle_probabilities)
    angles = rng.uniform(0.0, 2.0 * np.pi, size=n_points)
    sampled_radii = radii[circle_indices] * np.sqrt(rng.uniform(0.0, 1.0, size=n_points))
    offsets = np.column_stack((np.cos(angles), np.sin(angles))) * sampled_radii[:, None]
    return centers[circle_indices] + offsets


def sample_background_outliers(
    rng: np.random.Generator,
    n_points: int,
    *,
    bounds: np.ndarray,
    centers: np.ndarray,
    radii: np.ndarray,
) -> np.ndarray:
    if n_points == 0:
        return np.empty((0, 2), dtype=float)

    accepted_batches = []
    accepted_count = 0
    while accepted_count < n_points:
        batch_size = max(256, 4 * (n_points - accepted_count))
        candidates = np.column_stack(
            (
                rng.uniform(bounds[0, 0], bounds[0, 1], size=batch_size),
                rng.uniform(bounds[1, 0], bounds[1, 1], size=batch_size),
            )
        )
        distances = np.linalg.norm(candidates[:, None, :] - centers[None, :, :], axis=2)
        outside_all_disks = np.all(distances > radii[None, :], axis=1)
        accepted = candidates[outside_all_disks]
        if len(accepted) == 0:
            continue
        accepted = accepted[: n_points - accepted_count]
        accepted_batches.append(accepted)
        accepted_count += len(accepted)

    return np.vstack(accepted_batches)


rng = np.random.default_rng(RANDOM_SEED)

circle_points = sample_circle_boundaries(
    rng,
    centers=CIRCLE_CENTERS,
    radii=CIRCLE_RADII,
    points_per_circle=CIRCLE_POINTS_PER_CIRCLE,
    radial_jitter=CIRCLE_RADIAL_JITTER,
)
local_outlier_points = sample_circle_disk_outliers(
    rng,
    LOCAL_OUTLIER_POINTS,
    centers=CIRCLE_CENTERS,
    radii=CIRCLE_RADII,
)
background_outlier_points = sample_background_outliers(
    rng,
    BACKGROUND_OUTLIER_POINTS,
    bounds=OUTLIER_BOUNDS,
    centers=CIRCLE_CENTERS,
    radii=CIRCLE_RADII,
)
outlier_points = np.vstack((local_outlier_points, background_outlier_points))

points = np.vstack((circle_points, outlier_points))
labels = np.array([0] * CIRCLE_POINTS + [1] * OUTLIER_POINTS, dtype=int)

permutation = rng.permutation(len(points))
points = points[permutation]
labels = labels[permutation]

print("points shape:", points.shape)
print("circle points f=0:", int(np.sum(labels == 0)))
print("outlier points f=1:", int(np.sum(labels == 1)))
print("local outliers:", len(local_outlier_points))
print("background outliers outside all disks:", len(background_outlier_points))

## Assign Function Values

The function value is the chromatic label used by function-Alpha: blue noisy circle points have value `0`, and orange outlier/noise points have value `1`.

In [ ]:
function_values = labels.astype(float)

print("function min:", float(function_values.min()))
print("function max:", float(function_values.max()))
print("function values:", sorted(set(function_values.tolist())))

## Save the Point Cloud

The saved file is compatible with `pointcloud_to_scc2020.py` using the default `FUNCTION_MODE = "last"`.

In [ ]:
data = np.column_stack((points, function_values))

header_lines = [
    "# Chromatic four-circle point cloud with sparse outliers",
    "# columns: x y f",
    "# f: 0 for noisy circle points, 1 for outliers",
    f"# total_points: {len(points)}",
    f"# circle_points: {CIRCLE_POINTS}",
    f"# outlier_points: {OUTLIER_POINTS}",
    f"# local_outlier_points: {LOCAL_OUTLIER_POINTS}",
    f"# background_outlier_points: {BACKGROUND_OUTLIER_POINTS}",
    f"# circle_specs_center_x_center_y_radius: {CIRCLE_SPECS.tolist()}",
    f"# circle_points_per_circle: {CIRCLE_POINTS_PER_CIRCLE}",
    f"# circle_radial_jitter: {CIRCLE_RADIAL_JITTER}",
    f"# outlier_bounds_x_y: {OUTLIER_BOUNDS.tolist()}",
    "# outlier_sampling: local points in circle disks plus background points outside all disks",
    f"# random_seed: {RANDOM_SEED}",
]

rows = [f"{x:.17g} {y:.17g} {f_value:.17g}" for x, y, f_value in data]
OUTPUT_PATH.write_text("\n".join(header_lines + rows) + "\n", encoding="utf-8")

print("Saved:", OUTPUT_PATH)
print("Rows:", len(rows))

## Preview the Saved File

In [ ]:
saved_lines = OUTPUT_PATH.read_text(encoding="utf-8").splitlines()
print("\n".join(saved_lines[:20]))

## Visualize the Point Cloud

The plot uses blue circles for `f = 0` noisy circle points and orange squares for `f = 1` sparse outliers.

In [ ]:
import matplotlib.pyplot as plt

circle_mask = function_values == 0
outlier_mask = function_values == 1

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(
    points[circle_mask, 0],
    points[circle_mask, 1],
    s=28,
    marker="o",
    color="#3b4dff",
    edgecolor="#1f2fbf",
    linewidth=0.7,
    alpha=0.95,
    label="noisy circle points, f=0",
)
ax.scatter(
    points[outlier_mask, 0],
    points[outlier_mask, 1],
    s=26,
    marker="s",
    color="#ff914d",
    edgecolor="#d96a1e",
    linewidth=0.4,
    alpha=0.9,
    label="sparse outliers, f=1",
)

plot_padding = 0.04
min_xy = np.minimum(np.min(CIRCLE_CENTERS - CIRCLE_RADII[:, None], axis=0), OUTLIER_BOUNDS[:, 0]) - plot_padding
max_xy = np.maximum(np.max(CIRCLE_CENTERS + CIRCLE_RADII[:, None], axis=0), OUTLIER_BOUNDS[:, 1]) + plot_padding
ax.set_xlim(min_xy[0], max_xy[0])
ax.set_ylim(min_xy[1], max_xy[1])
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Chromatic four-circle point cloud")
ax.legend(loc="upper right", frameon=True)
plt.show()

## Use This File in the scc2020 Converter Notebook

In `Pointcloud_to_scc2020_tutorial.ipynb`, set:

```python
INPUT_POINTCLOUD = PROJECT_ROOT / "data" / "pointcloud_examples" / "chromatic_four_circles_500pts_function.txt"
FUNCTION_MODE = "last"
FILTRATION_METHOD = "alpha"  # or "rips"
```

Then run the conversion cells to save the corresponding `scc2020` chain complex.